# Merge Radiomics Features with Clinical Data
Join the radiomics features CSV with the clinical labels from bd_estudiUPF.csv.
Output: a merged table with study_id, radiomics features, and clinical outcome columns.

In [ ]:
import pandas as pd
import os

print("Imports OK")

In [ ]:
# load radiomics features
radiomics_path = os.path.join("reports", "12_radiomics_features_k3_i1.csv")
radiomics_df = pd.read_csv(radiomics_path)
print(f"Radiomics: {radiomics_df.shape[0]} rows, {radiomics_df.shape[1]} columns")

In [ ]:
# load clinical data
clinical_path = os.path.join("..", "data", "bd_estudiUPF.csv")
clinical_df = pd.read_csv(clinical_path)

# clean the study id column
clinical_df["id estudio"] = clinical_df["id estudio"].astype(str).str.strip()

print(f"Clinical: {clinical_df.shape[0]} rows, {clinical_df.shape[1]} columns")
print(f"Columns: {clinical_df.columns.tolist()}")

In [ ]:
# select the clinical columns we need
# primary label: RECHAZO CLINICO (binary 0/1)
# time context: motivo (1=1wk, 2=1mo, 3=1yr, 4=suspicion, 5=follow-up)
# patient grouping: id paciente (same patient may have multiple studies)
# secondary labels: biopsy-related columns
clinical_cols = [
    "id estudio",
    "id paciente",
    "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)=",
    "RECHAZO CLÍNICO",
    "Rechazo confirmadopor biopsia",
    "BIOPSIA",
]

# find biopsy rejection type column (name has inconsistent spacing in CSV)
biopsy_rej_cols = [c for c in clinical_df.columns if "RECHAZO biopsia" in c]
if biopsy_rej_cols:
    clinical_cols.append(biopsy_rej_cols[0])

# check which columns actually exist
available_cols = []
for col in clinical_cols:
    if col in clinical_df.columns:
        available_cols.append(col)
    else:
        print(f"WARNING: column not found: '{col}'")

clinical_subset = clinical_df[available_cols].copy()

# rename for easier use
rename_map = {
    "id estudio": "study_id",
    "id paciente": "patient_id",
    "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)=": "motivo",
    "RECHAZO CLÍNICO": "rejection",
    "Rechazo confirmadopor biopsia": "biopsy_confirmed_rejection",
    "BIOPSIA": "biopsy_performed",
}
if biopsy_rej_cols:
    rename_map[biopsy_rej_cols[0]] = "biopsy_rejection_type"

clinical_subset = clinical_subset.rename(columns=rename_map)
print(f"Clinical subset columns: {clinical_subset.columns.tolist()}")

In [ ]:
# inner join on study_id
merged = pd.merge(radiomics_df, clinical_subset, on="study_id", how="inner")
print(f"Merged: {merged.shape[0]} rows, {merged.shape[1]} columns")

# check for orphans
radiomics_ids = set(radiomics_df["study_id"])
clinical_ids = set(clinical_subset["study_id"])
in_radiomics_not_clinical = radiomics_ids - clinical_ids
in_clinical_not_radiomics = clinical_ids - radiomics_ids

if in_radiomics_not_clinical:
    print(f"In radiomics but not clinical: {in_radiomics_not_clinical}")
if in_clinical_not_radiomics:
    print(f"In clinical but not radiomics: {in_clinical_not_radiomics}")

In [ ]:
# class distribution
print("Rejection distribution (primary label):")
print(merged["rejection"].value_counts().to_string())
print()

print("Motivo distribution:")
print(merged["motivo"].value_counts().sort_index().to_string())
print()

print("Cross-tab: rejection by motivo")
print(pd.crosstab(merged["motivo"], merged["rejection"], margins=True))

In [ ]:
# save
output_path = os.path.join("reports", "13_merged_radiomics_clinical.csv")
merged.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Final shape: {merged.shape}")

merged.head(3)